# Fine-Tuning, LoRA & QLoRA — Google Colab Notebook

## Practical class notebook for intermediate students

This notebook explains and demonstrates **parameter-efficient fine-tuning** using:

- Full fine-tuning concept
- LoRA
- QLoRA
- Hugging Face Transformers
- PEFT
- bitsandbytes 4-bit quantization
- A small instruction dataset
- Adapter training
- Inference after fine-tuning

> This notebook is designed for teaching. It uses a small dataset so students can understand the complete workflow without waiting for long training.


## Learning Outcomes

By the end of this notebook, students should be able to:

1. Explain what fine-tuning is.
2. Explain why full fine-tuning is expensive.
3. Understand LoRA as a parameter-efficient fine-tuning method.
4. Understand QLoRA as quantized LoRA.
5. Prepare a small instruction dataset.
6. Load a base model.
7. Add LoRA adapters.
8. Train only adapter parameters.
9. Run inference after adapter training.
10. Understand common hyperparameters such as rank, alpha, dropout, learning rate, and batch size.


# 1. Theory Recap

## What is Fine-Tuning?

Fine-tuning means taking a **pre-trained model** and training it further on a **specific dataset**.

Example:

A general language model may already know English, programming, science, and general facts.  
If we train it on customer-support conversations, it can become better at answering support-related questions.

## Why not always use full fine-tuning?

Full fine-tuning updates **all model parameters**.

For large language models, this requires:

- Large GPU memory
- Long training time
- High cost
- Large model checkpoints
- More engineering effort

This is why **parameter-efficient fine-tuning** is useful.


# 2. LoRA Explanation

## What is LoRA?

**LoRA** stands for **Low-Rank Adaptation**.

Instead of updating all original weights of the model, LoRA freezes the base model and trains small additional matrices.

Simple idea:

```text
Original weight matrix W stays frozen.
LoRA adds small trainable matrices A and B.
Only A and B are updated during training.
```

Mathematical idea:

```text
W' = W + BA
```

Where:

- `W` = original frozen weight matrix
- `A` and `B` = small trainable low-rank matrices
- `r` = rank, controls adapter size


# 3. QLoRA Explanation

## What is QLoRA?

**QLoRA** means **Quantized LoRA**.

It combines:

```text
4-bit quantized base model + LoRA adapters
```

The base model is loaded in low memory using 4-bit quantization, while LoRA adapters are trained normally.

This allows larger models to be fine-tuned on limited GPU memory.

## Key idea

```text
LoRA = Freeze base model + train adapter weights
QLoRA = Quantize base model to 4-bit + train adapter weights
```


# 4. Colab Runtime Setup

Before running this notebook:

1. Go to **Runtime**
2. Click **Change runtime type**
3. Select **GPU**
4. Save

Then run the next cell to check the GPU.


In [22]:
!nvidia-smi

Sat Aug 15 08:09:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 5. Install Required Libraries

We need the following libraries:

- `transformers` for loading language models
- `datasets` for dataset handling
- `peft` for LoRA adapters
- `bitsandbytes` for QLoRA 4-bit quantization
- `accelerate` for efficient model loading
- `sentencepiece` and `protobuf` for tokenizer support


In [23]:
%pip install -q -U \
  transformers \
  datasets \
  peft \
  bitsandbytes \
  accelerate \
  sentencepiece \
  protobuf
%pip install pypdf

# 6. Import Libraries

In [24]:
import os
import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. QLoRA training may not work properly on CPU.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


# 7. Configuration

For classroom demonstration, we use a small instruction model.

You can change the base model later, but start small for faster testing.

## Modes

```python
USE_QLORA = True
```

This loads the base model in 4-bit and trains LoRA adapters.

```python
USE_QLORA = False
```

This loads the model normally and trains LoRA adapters without 4-bit quantization.


In [25]:
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

USE_QLORA = True

OUTPUT_DIR = "/content/lora_qlora_adapter"

MAX_LENGTH = 512

print("Base model:", BASE_MODEL)
print("Using QLoRA:", USE_QLORA)
print("Output directory:", OUTPUT_DIR)

Base model: Qwen/Qwen2.5-0.5B-Instruct
Using QLoRA: True
Output directory: /content/lora_qlora_adapter


# 8. Create a Small Instruction Dataset

Fine-tuning needs training examples.

Each example should contain:

```text
instruction/question → expected response
```

For this class, we create a small dataset about Generative AI, RAG, LoRA, QLoRA, and prompt engineering.

In real projects, your dataset should be much larger and domain-specific.


In [28]:
import os
import pandas as pd
from pypdf import PdfReader


# ============================================================
# 1. PDF FILE
# ============================================================

pdf_path = r"/content/ICTEcosystemPakistan.pdf"


# ============================================================
# 2. CHECK PDF EXISTS
# ============================================================

print("Checking PDF...")
print("Path:", pdf_path)

if not os.path.isfile(pdf_path):
    raise FileNotFoundError(
        f"\nPDF file not found:\n{pdf_path}\n\n"
        "Please check the file name and location."
    )

print("PDF found successfully!")


# ============================================================
# 3. READ PDF
# ============================================================

reader = PdfReader(pdf_path)

print(f"Number of pages: {len(reader.pages)}")


# ============================================================
# 4. EXTRACT TEXT
# ============================================================

pages_text = []

for page_number, page in enumerate(reader.pages, start=1):

    text = page.extract_text()

    if text:
        pages_text.append(text)

    print(
        f"Processed page {page_number} "
        f"of {len(reader.pages)}"
    )


full_text = "\n".join(pages_text)


print("\n====================================")
print("PDF extraction completed")
print("====================================")
print("Pages:", len(reader.pages))
print("Characters:", len(full_text))


# ============================================================
# 5. CLEAN TEXT
# ============================================================

full_text = full_text.replace("\x00", " ")

lines = []

for line in full_text.splitlines():

    line = line.strip()

    if line:
        lines.append(line)


clean_text = "\n".join(lines)


# ============================================================
# 6. SHOW SAMPLE TEXT
# ============================================================

print("\nFirst 1000 characters:")
print("------------------------------------")
print(clean_text[:1000])
print("------------------------------------")


# ============================================================
# 7. SPLIT INTO CHUNKS
# ============================================================

chunk_size = 1500

chunks = []

for i in range(0, len(clean_text), chunk_size):

    chunk = clean_text[i:i + chunk_size].strip()

    if chunk:
        chunks.append(chunk)


print("\nNumber of chunks:", len(chunks))


# ============================================================
# 8. CREATE TRAINING DATA
# ============================================================

training_data = []

for i, chunk in enumerate(chunks):

    training_data.append({
        "instruction": (
            "Based on the following information from "
            "the ICT Ecosystem Pakistan document, "
            "provide an accurate answer using only "
            "the information provided."
        ),
        "response": chunk
    })


# ============================================================
# 9. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(training_data)

print("\n====================================")
print("Dataset created")
print("====================================")
print("Training examples:", len(df))

display(df.head())

Checking PDF...
Path: /content/ICTEcosystemPakistan.pdf
PDF found successfully!
Number of pages: 7
Processed page 1 of 7
Processed page 2 of 7
Processed page 3 of 7
Processed page 4 of 7
Processed page 5 of 7
Processed page 6 of 7
Processed page 7 of 7

PDF extraction completed
Pages: 7
Characters: 31750

First 1000 characters:
------------------------------------
See discussions, stats, and author profiles for this publication at: https://www.researchgate.net/publication/317539647
Tracing ICT Innovation Ecosystem of Pakistan
Article · December 2016
CITATIONS
0
READS
470
2 authors, including:
Asif Ali Shah
Mehran University of Engineering and Technology
53 PUBLICATIONS   529 CITATIONS
SEE PROFILE
All content following this page was uploaded by Asif Ali Shah on 12 June 2017.
The user has requested enhancement of the downloaded file.

Abstract—Innovation Ecosystem plays a pivotal role in
facilitating activities that transform ideas into products,
processes and services. It is a complex 

,instruction,response
0,Based on the following information from the IC...,"See discussions, stats, and author profiles fo..."
1,Based on the following information from the IC...,onsor and\npromote creation of Startups. Acade...
2,Based on the following information from the IC...,o Pakistan. Email: nadeemprofessional@yahoo.co...
3,Based on the following information from the IC...,"wing its\nsignificance. However, some note wor..."
4,Based on the following information from the IC...,VIEW\nATTRIBUTES OF SUCCESSFUL INNOVATION ECOS...


In [ ]:
len(df)

# 9. Convert DataFrame to Hugging Face Dataset

In [29]:
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(
    test_size=0.15,
    seed=42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 17
})
Dataset({
    features: ['instruction', 'response'],
    num_rows: 4
})


# 10. Load Tokenizer

The tokenizer converts text into tokens.

For chat models, we format examples as:

```text
system message
user instruction
assistant response
```


In [30]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded.
Pad token: <|endoftext|>
EOS token: <|im_end|>


# 11. Format Dataset as Chat Examples

We convert each training row into a chat-style text format.

This helps the model learn how to respond as an assistant.


In [31]:
SYSTEM_MESSAGE = (
    "You are a helpful AI teacher. "
    "Answer clearly and simply for intermediate students."
)


def format_chat_example(example):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE
        },
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": text
    }


formatted_train = train_dataset.map(format_chat_example)
formatted_eval = eval_dataset.map(format_chat_example)

print(formatted_train[0]["text"])

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful AI teacher. Answer clearly and simply for intermediate students.<|im_end|>
<|im_start|>user
Based on the following information from the ICT Ecosystem Pakistan document, provide an accurate answer using only the information provided.<|im_end|>
<|im_start|>assistant
organizations
are playing their major role in driving whole system. First
category is comprises of funding organizations  in which
majority of the public and foreign funds for ICT related
scholarly research w ork are been channeled through HEC
which is a federally administered body. However, there are
some other federally controlled funding organizations working
within the ecosystem like USF (works under MoIT), PSF
(works under MoST) and National ICT R&D Fund ( works
under MoIT). Foreign funding is also been channeled through
these federally controlled organizations but some private
organizations (foreign and domestic) do sponsor projects
through organizing funding events. This category is

# 12. Tokenize Dataset

We convert text into token IDs.

For this teaching demo, we train the model to predict the full formatted text.  
In advanced production training, you may mask the prompt tokens and train only on assistant responses.


In [32]:
def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized


tokenized_train = formatted_train.map(
    tokenize_function,
    batched=False,
    remove_columns=formatted_train.column_names
)

tokenized_eval = formatted_eval.map(
    tokenize_function,
    batched=False,
    remove_columns=formatted_eval.column_names
)

print(tokenized_train)
print(tokenized_eval)

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 17
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})


# 13. Load Base Model

## For QLoRA

The model is loaded in **4-bit** using `BitsAndBytesConfig`.

## For LoRA

The model is loaded normally, and LoRA adapters are added.

> QLoRA requires GPU support. If you face issues, set `USE_QLORA = False` and restart runtime.


In [33]:
if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    model = prepare_model_for_kbit_training(model)

else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

model.config.use_cache = False

print("Base model loaded.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded.


# 14. Add LoRA Adapters

For Qwen-style models, common target modules are:

```text
q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
```

These are attention and feed-forward projection layers where LoRA can adapt the model efficiently.


In [34]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

print("LoRA adapters added.")
model.print_trainable_parameters()

LoRA adapters added.
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


# 15. Training Arguments

Important hyperparameters:

| Hyperparameter | Meaning |
|---|---|
| `r` | LoRA rank / adapter size |
| `lora_alpha` | Scaling factor |
| `lora_dropout` | Dropout for adapter training |
| `learning_rate` | Step size during training |
| `batch_size` | Number of samples per step |
| `gradient_accumulation_steps` | Simulates larger batch size |
| `num_train_epochs` | Number of passes over dataset |


In [35]:
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=5,
    save_steps=10,
    save_total_limit=2,
    fp16=use_fp16,
    report_to="none",
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",
)

print("Training arguments ready.")

Training arguments ready.


# 16. Create Trainer

In [36]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("Trainer is ready.")

Trainer is ready.


# 17. Start Fine-Tuning

This is the actual adapter training step.

For classroom demo, this may take a few minutes depending on GPU.


In [37]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
5,3.370807,2.893852


TrainOutput(global_step=5, training_loss=3.357829523086548, metrics={'train_runtime': 9.7877, 'train_samples_per_second': 1.737, 'train_steps_per_second': 0.511, 'total_flos': 19150348615680.0, 'train_loss': 3.357829523086548, 'epoch': 1.0})

# 18. Save LoRA / QLoRA Adapter

Only adapter weights are saved.

This is much smaller than saving the full model.


In [43]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Adapter saved at:", OUTPUT_DIR)

Adapter saved at: /content/lora_qlora_adapter


In [44]:
from pathlib import Path

adapter_path = Path("/content/lora_qlora_adapter")

if adapter_path.exists():
    print("Adapter folder found.")
    print("Files inside adapter folder:")
    for file in adapter_path.iterdir():
        print("-", file.name)
else:
    print("Adapter folder not found. Please run the training and save-adapter cells first.")

Adapter folder found.
Files inside adapter folder:
- chat_template.jinja
- tokenizer_config.json
- README.md
- adapter_config.json
- tokenizer.json
- checkpoint-5
- adapter_model.safetensors


In [47]:
import shutil

adapter_folder = "/content/lora_qlora_adapter"
zip_output = "/content/lora_qlora_adapter"

shutil.make_archive(
    base_name=zip_output,
    format="zip",
    root_dir=adapter_folder
)

print("Adapter zipped successfully:")
print("/content/lora_qlora_adapter.zip")

Adapter zipped successfully:
/content/lora_qlora_adapter.zip


In [48]:
from google.colab import files

files.download("/content/lora_qlora_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 19. Test the Fine-Tuned Model

Now we ask the model questions.

The model should answer in the style of our small instruction dataset.


In [39]:
def generate_answer(question, max_new_tokens=180):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()


print(generate_answer("Explain QLoRA in simple words."))

QLoRA is short for Quantum LoRA, which stands for "Quantum Learning Regularization". It's an approach to learning that uses quantum mechanics principles to improve the performance of neural networks.

Think of a regular network (like a typical deep learning model) as having lots of tiny errors, like when you're playing tic-tac-toe - it gets stuck in a few places where it doesn't work very well. 

But QLoRA helps those tiny errors by making sure that all its parts are working together properly. Instead of trying to make each part do everything on its own, QLoRA encourages them to learn more from their neighbors instead.

It does this by using something called "quantum entanglement", which is like when two coins show different colors but at the same time. This makes the network think it has access to information about many things at once - much like how


# 20. Try More Questions

In [40]:
questions = [
    "What is MBBS",
    "What is the difference between LoRA and QLoRA?",
    "Why is full fine-tuning expensive?",
    "What is the role of rank r in LoRA?",
    "When should we use QLoRA?"
]

for question in questions:
    print("=" * 80)
    print("Question:", question)
    print("Answer:", generate_answer(question))
    print()

Question: What is MBBS
Answer: MBBS stands for Medical Students' Board of Examinations, the highest medical education examination conducted in India, for medical students.
It is also called as Jeevvy Vychodhni or JEE (Joint Entrance Examination) and it is an exam which is conducted by IIMs every year. It includes two parts - Physical Science and Medicine.
The syllabus consists of 60 subjects including Physics, Chemistry, Biology, Microbiology, Physiology, Anatomy & Surgical Sciences, General Surgery, Internal Medicine, Pediatrics, Orthopaedics & Neurology, Endocrinology, Urology, Dermatology, Ophthalmology, Paediatrics etc. The total duration of the exam is four hours.
This exam assesses both theoretical knowledge and practical skills of medical student.
In addition to this exam, there are some other exams like JEE Mains, JEE Advanced, J

Question: What is the difference between LoRA and QLoRA?
Answer: LORA stands for Long Range Association, while QLoRA stands for Quasi-Long Range Asso

# 21. Load Adapter Later

After training, you can reload the adapter later.

This is useful when you want to share only adapter weights with students.


In [41]:
# Example reload code.
# Run this in a fresh runtime after training if needed.

'''
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_PATH = "/content/lora_qlora_adapter"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
'''
print("Reload example is available in this cell.")

Reload example is available in this cell.


# 22. LoRA vs QLoRA Comparison

| Feature | LoRA | QLoRA |
|---|---|---|
| Base model precision | Usually 16-bit / 32-bit | 4-bit quantized |
| Trainable weights | LoRA adapters only | LoRA adapters only |
| Memory usage | Lower than full fine-tuning | Much lower |
| Training cost | Lower | Very low |
| Best for | Small to medium models | Larger models with limited GPU memory |
| Main trade-off | Needs more memory than QLoRA | Slight quality or speed trade-off possible |


# 23. Common Problems and Fixes

## Problem 1: CUDA out of memory

Fixes:

- Use smaller model
- Reduce `MAX_LENGTH`
- Reduce batch size
- Use QLoRA instead of LoRA
- Restart runtime

## Problem 2: bitsandbytes error

Fixes:

- Make sure Colab runtime has GPU enabled
- Restart runtime after installing packages
- Set `USE_QLORA = False` if only CPU is available

## Problem 3: Poor model response after training

Reasons:

- Dataset is too small
- Data quality is poor
- Training epochs are too few
- Prompt format is inconsistent

## Problem 4: Overfitting

Reasons:

- Dataset is too small
- Too many epochs
- Learning rate too high


# 24. Student Lab Task

## Scenario

You are building a small AI teaching assistant for a Generative AI course.

The assistant should answer questions about:

- Fine-tuning
- LoRA
- QLoRA
- RAG
- Prompt engineering
- Embeddings

## Required Tasks

1. Run this notebook in Google Colab.
2. Explain the difference between fine-tuning and prompting.
3. Train LoRA or QLoRA adapters using the given dataset.
4. Add 10 more examples to the dataset.
5. Run training again.
6. Ask 5 test questions.
7. Compare responses before and after adding more data.
8. Explain why QLoRA saves GPU memory.
9. Take screenshots of training output and generated answers.
10. Submit your short reflection.


# 25. Extension Task

Advanced students can improve this notebook by:

1. Using a larger domain-specific dataset.
2. Loading data from CSV.
3. Masking prompt tokens and training only on assistant responses.
4. Evaluating with ROUGE, BLEU, or human scoring.
5. Uploading adapter weights to Hugging Face Hub.
6. Testing a larger model if GPU memory allows.


# Final Summary

Fine-tuning adapts a model to a specific task or domain.

LoRA makes fine-tuning efficient by training small adapter matrices instead of the full model.

QLoRA makes it even more memory-efficient by loading the base model in 4-bit precision while training LoRA adapters.

This notebook demonstrates the practical workflow:

```text
Prepare dataset
Load tokenizer
Load base model
Apply LoRA / QLoRA
Train adapters
Save adapters
Run inference
Evaluate results
```
